# TorchTitan-NPU Profiling：Varlen+FSDP 4096 vs Varlen+CP 8192

本节比较等 token 工作量的 Varlen+FSDP 4096 与 Varlen+CP 8192。

> 某组 trace 缺失时会标记为 `missing`；运行对应采集单元后重新分析即可。若 trace 是旧版按 EOS 判断样本边界的实现生成的，也必须重采，不能拿来评价当前实现。

## 对比对象与控制变量

| 实验 | trace 目录 | seq len | global batch | 容器 token/step |
|---|---|---:|---:|---:|
| Varlen+FSDP | `08_varlen_fsdp_s4096_positions_v2` | 4096 | 4 | 16384 |
| Varlen+CP | `08_varlen_cp_s8192_positions_v2` | 8192 | 2 | 16384 |

两组都使用两张 NPU、DataLoader local batch 2 和相同 profiling 窗口，并采集两个 rank；每卡在进入 Attention 前都持有 8192 个 local token。配置中的 `profile_step_start=5` 对应训练日志的 Step 5；Ascend 导出的 `step_trace_time.csv` 会把同一批次显示为零起始的 `Step=4`。

使用 TorchTitan-NPU 仓库内的 `torchtitan_npu.models.qwen3.config_registry.sft_qwen3_1_7b_wordle_tnd`，设置 `dp_shard=2, cp=1, seq_len=4096, GBS=4`。若当前分支没有该配置，请先按 08.02 中的说明配置 TorchTitan-NPU clone（`config_registry.py` 中的 `sft_qwen3_1_7b_wordle_tnd`）。

In [ ]:
import os
from pathlib import Path

original_dir = Path.cwd()
configured_root = os.environ.get('TORCHTITAN_ROOT')
candidates = [Path(configured_root)] if configured_root else []
for parent in (original_dir, *original_dir.parents):
    candidates.extend((parent / 'torchtitan-npu', parent.parent / 'torchtitan-npu'))
torchtitan_root = next((path.resolve() for path in candidates if (path / 'scripts/run_train.sh').is_file()), None)
if torchtitan_root is None:
    raise RuntimeError('未找到 torchtitan-npu；请设置 TORCHTITAN_ROOT。')
os.chdir(torchtitan_root)
if cann_env := os.environ.get('CANN_ENV_SCRIPT'):
    os.environ['BASH_ENV'] = cann_env
print('torchtitan root:', torchtitan_root)

In [ ]:
%%bash
set -euo pipefail
# 当前 position-boundary 证据使用独立目录，避免与旧 EOS trace 混淆。
rm -rf outputs/checkpoints/08_varlen_fsdp_s4096_positions_v2 outputs/profile_traces/08_varlen_fsdp_s4096_positions_v2
HCCL_IF_BASE_PORT=32010 NGPU=2 \
DATASET_PATH=./assets/data/wordle \
MODULE=torchtitan_npu.models.qwen3 \
CONFIG=sft_qwen3_1_7b_wordle_tnd \
bash scripts/run_train.sh \
  --training.steps 10 \
  --training.global-batch-size 4 \
  --checkpoint.folder checkpoints/08_varlen_fsdp_s4096_positions_v2 \
  --training.seq_len 4096 \
  --parallelism.data_parallel_replicate_degree 1 \
  --parallelism.data_parallel_shard_degree 2 \
  --parallelism.context_parallel_degree 1 \
  --profiling.enable-profiling \
  --profiling.profile-ranks -1 \
  --profiling.profile-step-start 5 \
  --profiling.profile-step-end 6 \
  --profiling.profile-with-memory \
  --profiling.save-traces-folder profile_traces/08_varlen_fsdp_s4096_positions_v2 \
dataloader:chat_data_loader_config \
  --dataloader.dataset_path "assets/data/wordle"


## 快速检查两组 Ascend Profiler 输出

分析单元读取：

- `step_trace_time.csv`：captured step 的 stage、compute、communication 与 overlap；
- `communication.json`：all-reduce、all-gather、reduce-scatter、all-to-all 等 collective 的次数、耗时、等待和传输量；
- `operator_details.csv`：FlashAttention 前向/反向的 device time 与 attention 输入 shape；
- `memory_record.csv`：采集窗口内每个 rank 的 peak active memory。

`communication.json` 的不同链路层可能重复记录同一份 payload，因此代码为每个 collective 取各链路 `Transit Size(MB)` 的最大值，不把 HCCS/SIO/SDMA 简单相加。这里逐 rank 做采集完整性和执行路径检查；双 rank 关键路径与工作量归因统一放在 08.04。

In [ ]:
from __future__ import annotations

import csv
import json
import re
from collections import Counter, defaultdict
from pathlib import Path

PROFILE_ROOT = Path('./outputs/profile_traces')
RUNS = {
    f'{route}_rank{rank}': {
        'route': route,
        'rank': rank,
        'label': f'{label} rank {rank}',
        'root': PROFILE_ROOT / folder,
    }
    for route, label, folder in (
        ('varlen_fsdp', 'Varlen+FSDP 4096', '08_varlen_fsdp_s4096_positions_v2'),
        ('varlen_cp', 'Varlen+CP 8192', '08_varlen_cp_s8192_positions_v2'),
    )
    for rank in (0, 1)
}
REQUIRED_FILES = (
    'step_trace_time.csv',
    'communication.json',
    'operator_details.csv',
    'memory_record.csv',
)


def number(value) -> float:
    try:
        return float(str(value).strip().replace(',', ''))
    except (TypeError, ValueError):
        return 0.0


def read_csv(path: Path) -> list[dict[str, str]]:
    with path.open(encoding='utf-8-sig', newline='') as handle:
        return list(csv.DictReader(handle))


def locate_export(root: Path, rank: int) -> tuple[Path | None, str]:
    if not root.exists():
        return None, f'missing: {root}'
    exports = sorted(
        path for path in root.glob('**/ASCEND_PROFILER_OUTPUT')
        if path.is_dir() and (path / f'ascend_pytorch_profiler_{rank}.db').is_file()
    )
    if len(exports) != 1:
        return None, f'rank {rank}: expected one ASCEND_PROFILER_OUTPUT, found {len(exports)}'
    missing = [name for name in REQUIRED_FILES if not (exports[0] / name).is_file()]
    if missing:
        return None, 'incomplete: ' + ', '.join(missing)
    return exports[0], 'ready'


def mean_column(rows: list[dict[str, str]], name: str) -> float:
    values = [number(row.get(name)) for row in rows]
    return sum(values) / len(values) if values else 0.0


def classify_collective(name: str) -> str:
    compact = re.sub(r'[^a-z]', '', name.lower())
    if 'reducescatter' in compact:
        return 'reduce-scatter'
    if 'allreduce' in compact:
        return 'all-reduce'
    if 'allgather' in compact:
        return 'all-gather'
    if 'alltoall' in compact:
        return 'all-to-all'
    if 'broadcast' in compact:
        return 'broadcast'
    if 'receive' in compact or compact.endswith('recv'):
        return 'receive'
    if 'send' in compact:
        return 'send'
    return 'other'


def summarize_communication(path: Path) -> dict[str, dict[str, float]]:
    payload = json.loads(path.read_text(encoding='utf-8'))
    totals = defaultdict(lambda: {'calls': 0, 'elapsed_ms': 0.0, 'wait_ms': 0.0, 'transit_mb': 0.0})
    for step in payload.values():
        for section in ('collective', 'p2p'):
            for event_name, event in step.get(section, {}).items():
                if event_name == 'Total Op Info':
                    continue
                kind = classify_collective(event_name)
                time_info = event.get('Communication Time Info', {})
                bandwidth_info = event.get('Communication Bandwidth Info', {})
                link_sizes = [
                    number(link.get('Transit Size(MB)'))
                    for link in bandwidth_info.values()
                    if isinstance(link, dict)
                ]
                totals[kind]['calls'] += 1
                totals[kind]['elapsed_ms'] += number(time_info.get('Elapse Time(ms)'))
                totals[kind]['wait_ms'] += number(time_info.get('Wait Time(ms)'))
                totals[kind]['transit_mb'] += max(link_sizes, default=0.0)
    return dict(totals)


def compact_shape(value: str, limit: int = 100) -> str:
    value = ' '.join(str(value).replace(';', ' | ').split())
    return value if len(value) <= limit else value[:limit - 3] + '...'


def summarize_attention(rows: list[dict[str, str]]) -> tuple[dict[str, dict[str, float]], Counter]:
    kernels = defaultdict(lambda: {'calls': 0, 'device_ms': 0.0})
    shapes = Counter()
    for row in rows:
        name = str(row.get('Name', ''))
        lower = name.lower()
        if name.startswith('aclnn') and 'attention' in lower:
            kernels[name]['calls'] += 1
            kernels[name]['device_ms'] += number(row.get('Device Total Duration(us)')) / 1000
        if 'scaled_dot_product_attention' in lower or 'fusion_attention' in lower:
            shape = compact_shape(row.get('Input Shapes', ''))
            if shape:
                shapes[(name, shape)] += 1
    return dict(kernels), shapes


def summarize_run(key: str, config: dict) -> dict:
    export, status = locate_export(config['root'], config['rank'])
    report = {**config, 'key': key, 'status': status, 'export': export}
    if export is None:
        return report

    step_rows = read_csv(export / 'step_trace_time.csv')
    memory_rows = read_csv(export / 'memory_record.csv')
    operator_rows = read_csv(export / 'operator_details.csv')
    communication = summarize_communication(export / 'communication.json')
    attention, attention_shapes = summarize_attention(operator_rows)
    communication_ms = mean_column(step_rows, 'Communication') / 1000
    overlapped_ms = mean_column(step_rows, 'Overlapped') / 1000

    report.update({
        'captured_steps': ','.join(str(row.get('Step', '?')) for row in step_rows),
        'stage_ms': mean_column(step_rows, 'Stage') / 1000,
        'compute_ms': mean_column(step_rows, 'Computing') / 1000,
        'communication_ms': communication_ms,
        'unoverlapped_ms': mean_column(step_rows, 'Communication(Not Overlapped)') / 1000,
        'overlap_pct': 100 * overlapped_ms / communication_ms if communication_ms else 0.0,
        'peak_active_mb': max((number(row.get('Total Active(MB)')) for row in memory_rows), default=0.0),
        'communication': communication,
        'attention': attention,
        'attention_shapes': attention_shapes,
    })
    return report


def table(headers: list[str], rows: list[list[str]]) -> None:
    rows = [[str(value) for value in row] for row in rows]
    widths = [max(len(headers[i]), *(len(row[i]) for row in rows)) for i in range(len(headers))]
    line = lambda row: '| ' + ' | '.join(row[i].ljust(widths[i]) for i in range(len(headers))) + ' |'
    print(line(headers))
    print('|-' + '-|-'.join('-' * width for width in widths) + '-|')
    for row in rows:
        print(line(row))


reports = {key: summarize_run(key, config) for key, config in RUNS.items()}

print('Profile discovery')
table(
    ['run', 'status', 'export'],
    [
        [report['label'], report['status'], report['export'] or report['root']]
        for report in reports.values()
    ],
)

ready = [report for report in reports.values() if report['status'] == 'ready']
if ready:
    print('\nCaptured-step breakdown (each rank; averages when multiple rows exist)')
    table(
        ['run', 'step', 'stage ms', 'compute ms', 'comm ms', 'unoverlap ms', 'overlap %', 'peak active MB'],
        [
            [
                report['label'], report['captured_steps'], f"{report['stage_ms']:.2f}",
                f"{report['compute_ms']:.2f}", f"{report['communication_ms']:.2f}",
                f"{report['unoverlapped_ms']:.2f}", f"{report['overlap_pct']:.1f}",
                f"{report['peak_active_mb']:.1f}",
            ]
            for report in ready
        ],
    )

    communication_rows = []
    for report in ready:
        for kind, values in sorted(report['communication'].items()):
            communication_rows.append([
                report['label'], kind, values['calls'], f"{values['elapsed_ms']:.2f}",
                f"{values['wait_ms']:.2f}", f"{values['transit_mb']:.2f}",
            ])
    print('\nCollective/P2P breakdown from communication.json')
    table(['run', 'kind', 'calls', 'elapsed ms', 'wait ms', 'transit MB'], communication_rows)

    attention_rows = []
    for report in ready:
        if not report['attention']:
            attention_rows.append([report['label'], '<none>', 0, '0.00'])
        for name, values in sorted(report['attention'].items()):
            attention_rows.append([report['label'], name, values['calls'], f"{values['device_ms']:.2f}"])
    print('\nAttention leaf APIs from operator_details.csv')
    table(['run', 'API', 'calls', 'device total ms'], attention_rows)

    print('\nObserved attention input shapes (top 6 per run)')
    for report in ready:
        print(f"\n{report['label']}")
        for (name, shape), count in report['attention_shapes'].most_common(6):
            print(f'  {count:>3} x {name}: {shape}')

print('\nMechanism checks')
for route, collective in (('varlen_fsdp', 'all-gather'), ('varlen_cp', 'all-to-all')):
    route_reports = [report for report in reports.values() if report['route'] == route]
    for report in route_reports:
        if report['status'] != 'ready':
            print(f"  [NOT MEASURED] {report['label']}: {report['status']}")
            continue
        calls = report['communication'].get(collective, {}).get('calls', 0)
        attention_calls = sum(item['calls'] for item in report['attention'].values())
        tnd_shapes = [
            shape for (name, shape) in report['attention_shapes']
            if 'fusion_attention' in name.lower()
            and len(re.findall(r'\d+', shape.split('|', 1)[0])) == 3
        ]
        state = 'PASS' if calls and attention_calls and tnd_shapes else 'WARN'
        print(
            f'  [{state}] {report["label"]}: {collective} calls = {calls}; '
            f'attention leaf calls = {attention_calls}; TND-like shapes = {len(tnd_shapes)}'
        )

missing = [report['label'] for report in reports.values() if report['status'] != 'ready']
if missing:
    print('\nRun the corresponding profiling notebook(s), then rerun this cell: ' + ', '.join(missing))

## 对比结果解读

2026-07-29 本轮 `_positions_v2` 重采已同时找到四个 rank 输出。FSDP rank 0/1 的 trace 区间数为 `11/5`，CP 两个 rank 均为 `16`，全部与当前位置编号重放一致；CP trace 的 TND Q shape 为 `[16384,8,128]`。因此这组 trace 可以进入 08.04 的性能归因。

两组都使用 Varlen/TND，主要比较 FSDP 的 all-gather/reduce-scatter 与 CP 新增的 Q/K/V/output all-to-all。关键路径以 `step_trace_time.csv` 的 `Communication(Not Overlapped)` 为准；`communication.json` 中各 event elapsed 可能包含等待和并发，不能直接相加。

采样 step 的慢 rank Stage 为 FSDP `1658.489 ms`、CP `1788.742 ms`。这个数字只说明 CP 的物理 step 在本次 trace 中更慢；raw sample 和 supervised token 还要结合 DataLoader 计数，并用关闭 profiler 的稳定运行确认。完整结果见 08.04。


In [ ]:
%cd $original_dir


## 练习

1. （判断题）只采 rank 0 的 trace 就足以判断双卡训练的关键路径和 rank skew。

2. （判断题）Profiler 适合确认算子与通信路径；稳定吞吐应另用关闭 profiler 的重复运行统计。

3. （单选题）比较 captured step 的 Stage、Computing 和 Communication(Not Overlapped) 时，主要读取哪个文件？
    A. step_trace_time.csv
    B. tokenizer_config.json
    C. memory_record.csv
    D. checkpoint metadata

4. （多选题）一次可审计的重采应保存哪些内容？
    A. 双 rank trace
    B. 独立且不覆盖旧证据的输出目录
    C. 对应配置和训练日志
    D. 只截取终端最后一行

In [ ]:
!cat ./answer/08.03_answer.txt
